# get historical pricing elektriciteit

Deze data is beschikbaar op https://opendata.elia.be/explore/dataset/ods134/table/?q.timerange.datetime=datetime:%5B2024-10-31T23:00:00Z%20TO%202026-03-15T22:59:59Z%5D&sort=datetime

Wordt bewaard in pricing.csv

# get historical weather data

In [3]:
import os

In [5]:
os.getcwd()

'c:\\Users\\kurtm\\Documents\\Eindewerk'

In [7]:
import requests
import pandas as pd
from datetime import datetime

# =============================================
# CONFIG - parameters
# =============================================

lat = 50.9281      # Vilvoorde
lon = 4.4191

start_date = "2024-05-22"   # kleine testperiode eerst!
end_date   = "2026-03-14"

StoreDirectory = "c:\\Users\\kurtm\\Documents\\Eindewerk\\V1\\Source Data"

# Veilige variabelen (geen tilt nodig → geen risico op fout)
hourly_vars = [
    "shortwave_radiation",       # GHI ≈ global horizontal (W/m², preceding hour mean)
    "direct_radiation",
    "diffuse_radiation",
    "sunshine_duration",         # seconden zonneschijn vorige uur
    "direct_normal_irradiance",  # DNI
]

# Als je toch getilde instraling wilt → uncomment + vul tilt/azimuth in
hourly_vars.append("global_tilted_irradiance")
# extra_params = {"tilt": 35, "azimuth": 180}   # zuid = 180°

# =============================================

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": ",".join(hourly_vars),
    "timezone": "Europe/Brussels",
    # ** voeg toe als je global_tilted_irradiance vraagt **
    # "tilt": 35,
    # "azimuth": 180,
}

print(f"Request naar: {url}")
print("Params:", params)

response = requests.get(url, params=params)
print(f"Status code: {response.status_code}")

if response.status_code != 200:
    print("API gaf geen 200 →", response.text)
    exit()

try:
    data = response.json()
except ValueError:
    print("Geen valide JSON →", response.text)
    exit()

# Debug: toon hele response (belangrijk!)
print("\nVolledige API response:")
print(data)

if "error" in data:
    print("API error:", data.get("reason", "Onbekend"))
    exit()

if "hourly" not in data:
    print("Geen 'hourly' key in response. Mogelijk ongeldige variabele of geen data.")
    exit()

# Nu veilig DataFrame bouwen
df = pd.DataFrame({
    "time": data["hourly"]["time"],
    **{var: data["hourly"].get(var, [None] * len(data["hourly"]["time"])) for var in hourly_vars}
})

df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time")

# Extra kolom: zonneschijn in minuten per uur
if "sunshine_duration" in df.columns:
    df["sunshine_min_per_hour"] = df["sunshine_duration"] / 60

print("\nEerste 10 rijen:")
print(df.head(10))

print(f"\nTotaal rijen: {len(df)}")

csv_file = os.path.join(StoreDirectory,f"vilvoorde_zonneschijn.csv")
df.to_csv(csv_file, float_format="%.2f")
print(f"Opgeslagen: {csv_file}")

Request naar: https://archive-api.open-meteo.com/v1/archive
Params: {'latitude': 50.9281, 'longitude': 4.4191, 'start_date': '2024-05-22', 'end_date': '2026-03-14', 'hourly': 'shortwave_radiation,direct_radiation,diffuse_radiation,sunshine_duration,direct_normal_irradiance,global_tilted_irradiance', 'timezone': 'Europe/Brussels'}
Status code: 200

Volledige API response:
{'latitude': 50.931458, 'longitude': 4.339286, 'generationtime_ms': 49.71182346343994, 'utc_offset_seconds': 3600, 'timezone': 'Europe/Brussels', 'timezone_abbreviation': 'GMT+1', 'elevation': 15.0, 'hourly_units': {'time': 'iso8601', 'shortwave_radiation': 'W/m²', 'direct_radiation': 'W/m²', 'diffuse_radiation': 'W/m²', 'sunshine_duration': 's', 'direct_normal_irradiance': 'W/m²', 'global_tilted_irradiance': 'W/m²'}, 'hourly': {'time': ['2024-05-22T00:00', '2024-05-22T01:00', '2024-05-22T02:00', '2024-05-22T03:00', '2024-05-22T04:00', '2024-05-22T05:00', '2024-05-22T06:00', '2024-05-22T07:00', '2024-05-22T08:00', '202

# verbruik downloaden - komt van Fluvius